# 03 — Feature Engineering

**Input:** `../data/processed/pm_day_clean.csv`, `../data/processed/embeddings.npy`
**Output:** `../data/processed/pm_day_features.csv`

**Description:**
- Compute PCA on embeddings (content representation)
- Compute engagement features (word count, log word count, minimal text flag)
- Compute lexical constriction features (type-token ratio, root TTR)
- Compute within-person instability (cosine distance to person centroid)
- Decompose all features into within-person and between-person components
- Save feature-enriched dataset for modeling

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "processed", "pm_day_clean.csv")
EMBED_PATH = os.path.join("..", "data", "processed", "embeddings.npy")
OUT_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")

PID_COL = "expiwell_id_clean"
TEXT_COL = "pm_day_text"
CRISIS_COL = "crisis_PM_from_full"

N_PCS = 20
N_PCS_CONTENT = 5
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

In [ ]:
# =========================
# LOAD
# =========================
pm_day = pd.read_csv(DATA_PATH)
X_text = np.load(EMBED_PATH)

assert X_text.shape[0] == len(pm_day),     f"Embedding rows ({X_text.shape[0]}) != data rows ({len(pm_day)})"

print("Loaded data:", pm_day.shape)
print("Loaded embeddings:", X_text.shape)

In [ ]:
# =========================
# ENGAGEMENT FEATURES
# =========================
txt = pm_day[TEXT_COL].fillna("").astype(str)
pm_day["word_count"] = txt.str.split().map(len).astype(float)
pm_day["log1p_wc"] = np.log1p(pm_day["word_count"])
pm_day["wc_le1"] = (pm_day["word_count"] <= 1).astype(float)

print("Engagement features computed.")
print("  Mean word count:", pm_day["word_count"].mean().round(1))
print("  % minimal text (<=1 word):", pm_day["wc_le1"].mean().round(3))

In [ ]:
# =========================
# LEXICAL CONSTRICTION FEATURES
# =========================
# Theoretical grounding: Shneidman's cognitive constriction predicts that
# crisis states narrow the range of perceived options, which manifests
# linguistically as reduced vocabulary diversity and increased word repetition.
#
# TTR (type-token ratio): unique words / total words.
#   - Higher = more diverse vocabulary; lower = more repetitive/constricted.
#   - Known to be length-dependent (shorter texts have higher TTR mechanically).
#
# Root TTR (Guiraud's index): unique words / sqrt(total words).
#   - Partially corrects for text length dependence.
#   - More stable across varying response lengths in EMA.
#
# Within-person centering (computed later) gives the critical interpretation:
# when a person's TTR drops below their own average, their vocabulary is
# narrowing relative to their typical expression -- a state-level constriction signal.

def compute_lexical_features(text_series):
    ttr_vals = []
    root_ttr_vals = []
    n_unique_vals = []

    for text in text_series:
        text = str(text).strip()
        words = text.lower().split()
        n_total = len(words)

        if n_total <= 1:
            # Cannot compute meaningful diversity from 0-1 words
            ttr_vals.append(np.nan)
            root_ttr_vals.append(np.nan)
            n_unique_vals.append(np.nan)
        else:
            n_unique = len(set(words))
            ttr_vals.append(n_unique / n_total)
            root_ttr_vals.append(n_unique / np.sqrt(n_total))
            n_unique_vals.append(float(n_unique))

    return ttr_vals, root_ttr_vals, n_unique_vals

ttr, root_ttr, n_unique = compute_lexical_features(pm_day[TEXT_COL].fillna(""))

pm_day["ttr"] = ttr
pm_day["root_ttr"] = root_ttr
pm_day["n_unique_words"] = n_unique

# Descriptives
valid = pm_day["ttr"].notna()
print("Lexical constriction features computed.")
print(f"  Valid observations (wc > 1): {valid.sum()} / {len(pm_day)}")

ttr_m = pm_day.loc[valid, "ttr"]
rttr_m = pm_day.loc[valid, "root_ttr"]
print(f"  TTR:      mean={ttr_m.mean():.3f}  sd={ttr_m.std():.3f}  range=[{ttr_m.min():.3f}, {ttr_m.max():.3f}]")
print(f"  Root TTR: mean={rttr_m.mean():.3f}  sd={rttr_m.std():.3f}  range=[{rttr_m.min():.3f}, {rttr_m.max():.3f}]")

# Check correlation with word count (expected: negative -- longer texts have lower TTR)
wc_ttr_corr = pm_day.loc[valid, ["word_count", "ttr"]].corr().iloc[0, 1]
wc_rttr_corr = pm_day.loc[valid, ["word_count", "root_ttr"]].corr().iloc[0, 1]
print(f"  corr(word_count, TTR) = {wc_ttr_corr:.3f}  (expected: negative)")
print(f"  corr(word_count, root_TTR) = {wc_rttr_corr:.3f}  (should be less negative)")

In [ ]:
# =========================
# PCA ON EMBEDDINGS
# =========================
def l2_normalize_rows(X, eps=1e-12):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(n, eps)

X_norm = l2_normalize_rows(X_text)

pca_full = PCA(n_components=N_PCS, random_state=RANDOM_SEED)
X_pcs = pca_full.fit_transform(X_text)

for k in range(N_PCS):
    pm_day[f"PC{k+1}"] = X_pcs[:, k]

print(f"PCA: {N_PCS} components")
print("  Variance explained (first 5):", np.round(pca_full.explained_variance_ratio_[:5], 4))
print("  Cumulative variance ({} PCs): {:.3f}".format(N_PCS, pca_full.explained_variance_ratio_.sum()))

In [ ]:
# =========================
# SCREE PLOT + VARIANCE EXPLAINED
# =========================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Scree plot (individual variance per component)
axes[0].bar(range(1, N_PCS + 1), pca_full.explained_variance_ratio_, color="steelblue", alpha=0.8)
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Proportion of Variance Explained")
axes[0].set_title("A. Individual Variance per Component")
axes[0].set_xticks(range(1, N_PCS + 1))

# Panel B: Cumulative variance
cumvar = pca_full.explained_variance_ratio_.cumsum()
axes[1].plot(range(1, N_PCS + 1), cumvar, "o-", color="steelblue", linewidth=2)
axes[1].axhline(y=0.50, linestyle="--", color="gray", alpha=0.7, label="50%")
axes[1].axhline(y=0.80, linestyle="--", color="gray", alpha=0.4, label="80%")
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Variance Explained")
axes[1].set_title("B. Cumulative Variance")
axes[1].set_xticks(range(1, N_PCS + 1))
axes[1].legend()

plt.tight_layout()

fig_path = os.path.join("..", "outputs", "figures", "pca_scree.png")
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {fig_path}")
print("")
print("Variance explained per component:")
for i in range(N_PCS):
    print(f"  PC{i+1:2d}: {pca_full.explained_variance_ratio_[i]:.4f}  cumulative: {cumvar[i]:.4f}")

In [ ]:
# =========================
# WITHIN-PERSON INSTABILITY
# =========================
pid = pm_day[PID_COL].astype(str).values
centroids = np.zeros_like(X_norm, dtype=float)

for p in np.unique(pid):
    idx = np.where(pid == p)[0]
    c = X_norm[idx].mean(axis=0)
    nrm = np.linalg.norm(c)
    if nrm > 0:
        c = c / nrm
    centroids[idx] = c

pm_day["instability_cosdist"] = 1.0 - np.sum(X_norm * centroids, axis=1)

print("Instability descriptives:")
print(pm_day["instability_cosdist"].describe())

In [ ]:
# =========================
# WITHIN vs BETWEEN DECOMPOSITION
# =========================
# Include constriction features in the decomposition
MECH_COLS = (
    ["log1p_wc", "wc_le1", "instability_cosdist", "ttr", "root_ttr"] +
    [f"PC{i}" for i in range(1, N_PCS_CONTENT + 1)]
)

g = pm_day.groupby(PID_COL)
for c in MECH_COLS:
    pm_day[f"{c}_between"] = g[c].transform("mean")
    pm_day[f"{c}_within"] = pm_day[c] - pm_day[f"{c}_between"]

print("Within/between decomposition computed for:", MECH_COLS)

# Quick check: within-person TTR variance
ttr_valid = pm_day["ttr_within"].dropna()
print("")
print("Within-person TTR variation:")
print(f"  SD of ttr_within: {ttr_valid.std():.4f}")
print(f"  Range: [{ttr_valid.min():.3f}, {ttr_valid.max():.3f}]")

# Correlation of within-person TTR drop with crisis
crisis_valid = pm_day[["ttr_within", CRISIS_COL]].dropna()
if len(crisis_valid) > 50:
    r = crisis_valid.corr().iloc[0, 1]
    print(f"  corr(ttr_within, crisis) = {r:.4f}")
    direction = "negative = constriction on high-crisis days" if r < 0 else "positive = more diverse on high-crisis days"
    print(f"  Interpretation: {direction}")

In [ ]:
# =========================
# SAVE
# =========================
pm_day.to_csv(OUT_PATH, index=False)
print("Saved feature dataset:", OUT_PATH)
print("Shape:", pm_day.shape)